# Load dataset 

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tokenize

In [2]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lalis\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 50000/50000 [00:10<00:00, 4755.73 examples/s]


# Load model and apply LoRA

In [4]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4903.04it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


# Metrics

In [5]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

# Training

In [6]:
from transformers import (
    TrainingArguments,
    Trainer,
)

training_args = TrainingArguments(
    output_dir="./imdb-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.308671,0.313937,0.866000
2,0.257027,0.291911,0.879000
3,0.238635,0.296815,0.878000


TrainOutput(global_step=1875, training_loss=0.2896548329671224, metrics={'train_runtime': 91.9271, 'train_samples_per_second': 326.346, 'train_steps_per_second': 20.397, 'total_flos': 2021091102720000.0, 'train_loss': 0.2896548329671224, 'epoch': 3.0})

# Save LoRA adapter

In [7]:
model.save_pretrained("./weights/imdb-lora-adapter")
tokenizer.save_pretrained("./weights/imdb-lora-adapter")

('./weights/imdb-lora-adapter\\tokenizer_config.json',
 './weights/imdb-lora-adapter\\tokenizer.json')

# Load later

In [9]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model = PeftModel.from_pretrained(
    base_model,
    "./weights/imdb-lora-adapter"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16666.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Inference 

### Load the saved model 

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

base_model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained("./weights/imdb-lora-adapter")

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

model = PeftModel.from_pretrained(
    base_model,
    "./weights/imdb-lora-adapter"
)

model.eval()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16650.01it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

### Predict sentiment

In [11]:
text = """
This movie was fantastic. The acting was excellent and
the story kept me engaged from beginning to end.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=-1).item()

label_map = {
    0: "negative",
    1: "positive"
}

print(label_map[prediction])

positive


### Get probabilities

In [12]:
import torch.nn.functional as F

with torch.no_grad():
    logits = model(**inputs).logits

probs = F.softmax(logits, dim=-1)

print("Negative:", probs[0][0].item())
print("Positive:", probs[0][1].item())

Negative: 0.003719477914273739
Positive: 0.9962804913520813


# Batch inference

In [13]:
texts = [
    "This movie was amazing!",
    "Worst film I have ever seen.",
    "It was okay, not great."
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    logits = model(**inputs).logits

preds = logits.argmax(dim=-1)

for text, pred in zip(texts, preds):
    print(text)
    print(label_map[pred.item()])
    print()

This movie was amazing!
positive

Worst film I have ever seen.
negative

It was okay, not great.
negative

